# 02 — Cleaning & Feature Engineering

Input: `philly311_raw.csv` from notebook 01.

**Before running:** open `philly311_raw.csv` and Step 2 of notebook 01's output, and fix
the column names in the CONFIG cell below so they match reality. Everything downstream
references these variables, so getting this right once saves you from hunting through
every cell later.

In [4]:
from google.colab import files
uploaded = files.upload()  # a file picker will pop up — choose philly311_raw.csv from your computer

Saving philly311_raw.csv to philly311_raw.csv


In [5]:
import pandas as pd
import numpy as np

# ---- CONFIG: confirm/edit these against your actual columns ----
CREATED_COL = "requested_datetime"   # candidates: requested_datetime, created_date
CLOSED_COL  = "closed_datetime"      # candidates: closed_datetime, closed_date
TYPE_COL    = "service_name"         # the request-type column you used in notebook 01
STATUS_COL  = "status"
ZIP_COL     = "zipcode"
AGENCY_COL  = "agency_responsible"
LAT_COL     = "lat"
LON_COL     = "lon"
ID_COL      = "objectid"             # or service_request_id, whichever is the unique id

df = pd.read_csv("philly311_raw.csv")
print(df.shape)
df.head()

/tmp/ipykernel_1132/937779364.py:15: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("philly311_raw.csv")


(275351, 23)


,the_geom,cartodb_id,the_geom_webmercator,objectid,service_request_id,status,status_notes,service_name,service_code,agency_responsible,...,expected_datetime,address,zipcode,media_url,lat,lon,subject,closed_datetime,gdb_geomattr_data,service_type
0,0101000020E61000002E47CD8588C552C0E2846BDEFA00...,1508,0101000020110F0000A260048FAEE25FC11E942CB8FB90...,1489,10038990,Closed,NaN,Abandoned Vehicle,SR-PD01,Police Department,...,2016-05-31 00:00:00+00,1926 AUTH ST,19124.0,NaN,40.007656,-75.086458,NaN,2016-01-11 18:07:11+00,NaN,Abandoned Vehicle
1,0101000020E61000002E47CD8588C552C0E2846BDEFA00...,4957,0101000020110F0000A260048FAEE25FC11E942CB8FB90...,4915,10043391,Closed,NaN,Abandoned Vehicle,SR-PD01,Police Department,...,2016-05-31 00:00:00+00,1926 AUTH ST,19124.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,40.007656,-75.086458,NaN,2016-01-11 18:05:52+00,NaN,Abandoned Vehicle
2,0101000020E610000016BC92C475CD52C0872070DFCCF9...,14738,0101000020110F000045C9926F25F05FC1BED6A2920689...,14655,10062003,Closed,Issue Resolved,Abandoned Vehicle,SR-PD01,Police Department,...,2016-05-31 00:00:00+00,4331 PINE ST,19104.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.951565,-75.210313,NaN,2016-01-21 16:16:55+00,NaN,Abandoned Vehicle
3,0101000020E610000024B209D602C952C01611424AFEFC...,14770,0101000020110F00006D2679D896E85FC19378385A908C...,14687,10062037,Closed,Issue Resolved,Abandoned Vehicle,SR-PD01,Police Department,...,2016-05-31 00:00:00+00,300 CECIL B MOORE AVE,19122.0,NaN,39.976510,-75.140798,NaN,2016-03-07 15:24:20+00,NaN,Abandoned Vehicle
4,0101000020E6100000D9F064C07BC752C04E9B174D44FD...,21893,0101000020110F00004CB8948CFEE55FC16A1170F2DD8C...,21787,10166284,Closed,Other,Abandoned Vehicle,SR-PD01,Police Department,...,2016-05-31 00:00:00+00,2636 ALMOND ST,19125.0,NaN,39.978647,-75.116928,NaN,2016-01-11 15:12:02+00,NaN,Abandoned Vehicle


## Step A — Parse dates and compute resolution time

In [6]:
df[CREATED_COL] = pd.to_datetime(df[CREATED_COL], errors="coerce")
df[CLOSED_COL] = pd.to_datetime(df[CLOSED_COL], errors="coerce")

df["resolution_hours"] = (df[CLOSED_COL] - df[CREATED_COL]).dt.total_seconds() / 3600

print("Rows with no created date:", df[CREATED_COL].isna().sum())
print("Rows with no closed date (i.e. still open, or never closed in this export):", df[CLOSED_COL].isna().sum())
print("Rows with negative resolution time (data errors):", (df["resolution_hours"] < 0).sum())

Rows with no created date: 0
Rows with no closed date (i.e. still open, or never closed in this export): 22932
Rows with negative resolution time (data errors): 0


## Step B — Decide how to handle open/invalid requests

Default policy below (edit and justify in your report if you change it):
- Drop rows with no created date (can't use them for anything)
- Drop rows with negative resolution time (data entry errors)
- Keep still-open rows in a *separate* dataframe rather than silently dropping them —
  they matter for context (e.g. "12% of requests are still open after 2 years") even
  though they can't be used as labeled examples for regression/classification.

In [7]:
df = df[df[CREATED_COL].notna()].copy()

still_open = df[df[CLOSED_COL].isna()].copy()
print(f"Still-open requests set aside: {len(still_open)} ({len(still_open)/len(df):.1%} of data)")

df = df[df[CLOSED_COL].notna()].copy()
n_before = len(df)
df = df[df["resolution_hours"] >= 0].copy()
print(f"Dropped {n_before - len(df)} rows with negative resolution time")
print(f"Remaining rows for modeling: {len(df)}")

Still-open requests set aside: 22932 (8.3% of data)
Dropped 0 rows with negative resolution time
Remaining rows for modeling: 252419


## Step C — Duplicates and missing key fields

In [8]:
dupes = df.duplicated(subset=[ID_COL]).sum()
print(f"Duplicate IDs: {dupes}")
df = df.drop_duplicates(subset=[ID_COL])

for col in [ZIP_COL, AGENCY_COL, LAT_COL, LON_COL]:
    if col in df.columns:
        pct_missing = df[col].isna().mean()
        print(f"{col}: {pct_missing:.1%} missing")

Duplicate IDs: 0
zipcode: 0.6% missing
agency_responsible: 0.0% missing
lat: 0.2% missing
lon: 0.2% missing


## Step D — Feature engineering

In [9]:
df["day_of_week"] = df[CREATED_COL].dt.day_name()
df["is_weekend"] = df[CREATED_COL].dt.dayofweek >= 5
df["hour_of_day"] = df[CREATED_COL].dt.hour
df["month"] = df[CREATED_COL].dt.month
df["year"] = df[CREATED_COL].dt.year

# log-transform target given the expected right skew (confirm this is warranted in notebook 03)
df["log_resolution_hours"] = np.log1p(df["resolution_hours"])

df.head()

,the_geom,cartodb_id,the_geom_webmercator,objectid,service_request_id,status,status_notes,service_name,service_code,agency_responsible,...,closed_datetime,gdb_geomattr_data,service_type,resolution_hours,day_of_week,is_weekend,hour_of_day,month,year,log_resolution_hours
0,0101000020E61000002E47CD8588C552C0E2846BDEFA00...,1508,0101000020110F0000A260048FAEE25FC11E942CB8FB90...,1489,10038990,Closed,NaN,Abandoned Vehicle,SR-PD01,Police Department,...,2016-01-11 18:07:11+00:00,NaN,Abandoned Vehicle,21.318333,Sunday,True,20,1,2016,3.105408
1,0101000020E61000002E47CD8588C552C0E2846BDEFA00...,4957,0101000020110F0000A260048FAEE25FC11E942CB8FB90...,4915,10043391,Closed,NaN,Abandoned Vehicle,SR-PD01,Police Department,...,2016-01-11 18:05:52+00:00,NaN,Abandoned Vehicle,21.187778,Sunday,True,20,1,2016,3.099542
2,0101000020E610000016BC92C475CD52C0872070DFCCF9...,14738,0101000020110F000045C9926F25F05FC1BED6A2920689...,14655,10062003,Closed,Issue Resolved,Abandoned Vehicle,SR-PD01,Police Department,...,2016-01-21 16:16:55+00:00,NaN,Abandoned Vehicle,258.710000,Sunday,True,21,1,2016,5.559566
3,0101000020E610000024B209D602C952C01611424AFEFC...,14770,0101000020110F00006D2679D896E85FC19378385A908C...,14687,10062037,Closed,Issue Resolved,Abandoned Vehicle,SR-PD01,Police Department,...,2016-03-07 15:24:20+00:00,NaN,Abandoned Vehicle,1358.188333,Monday,False,1,1,2016,7.214643
4,0101000020E6100000D9F064C07BC752C04E9B174D44FD...,21893,0101000020110F00004CB8948CFEE55FC16A1170F2DD8C...,21787,10166284,Closed,Other,Abandoned Vehicle,SR-PD01,Police Department,...,2016-01-11 15:12:02+00:00,NaN,Abandoned Vehicle,4.261111,Monday,False,10,1,2016,1.660342


## Step E — Save the clean dataset

In [10]:
df.to_csv("philly311_clean.csv", index=False)
still_open.to_csv("philly311_still_open.csv", index=False)
print("Saved philly311_clean.csv and philly311_still_open.csv")

Saved philly311_clean.csv and philly311_still_open.csv


## Write these down for your report
- How many rows you started with vs. ended with, and why each drop happened
- % still-open, and your reasoning for setting them aside
- Any missingness you noticed and what you did about it
- Anything surprising you saw while doing this (this is exactly the kind of thing your
  professor is asking about in "what problems did you encounter and how did you solve them")